# Modulo 5 — Validation

Il guardiano anti-overfitting. Ogni strategia passa per: **Walk-Forward** (profittevole in molte finestre temporali?), **Monte Carlo** (profittevole in molti riordinamenti dei trade?), **Robustness** (regge perturbando SL/TP?), **Sensitivity** (poco sensibile ai parametri?).

Una strategia è dichiarata **robusta** solo se supera TUTTE le soglie; altrimenti viene **scartata automaticamente**.

In [ ]:
import sys
from pathlib import Path
for c in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/trading-ai')]:
    if (c / 'trading_ai').exists():
        sys.path.insert(0, str(c)); break

from trading_ai.data_engine import DataEngine, generate_ohlcv
from trading_ai.feature_engineering import FeatureEngine
from trading_ai.pattern_discovery import PatternDiscovery
from trading_ai.strategy_generator import StrategyGenerator, RiskParams
from trading_ai.validation import Validator

In [ ]:
eng = DataEngine()
h1 = eng.to_timeframe(eng.load_dataframe(generate_ohlcv(n=250_000, seed=3)), 'H1')
feats = FeatureEngine().transform(h1, dropna=True)
disc = PatternDiscovery(n_clusters=25, horizon=10, min_profit_factor=1.05, min_count_test=15)
disc.discover(feats)
gen = StrategyGenerator(disc, risk=RiskParams(sl_atr=2, tp_atr=3, max_bars=40))
strategies = gen.build()
print('Strategie da validare:', len(strategies))

## Validazione completa

Il `Validator` restituisce, per ogni strategia, il verdetto e il motivo. Solo le strategie `robust=True` proseguono verso l'export EA (Modulo 6).

In [ ]:
v = Validator(min_trades=30, min_mc_prob_profit=0.60, min_wf_consistency=0.50,
              min_robust_fraction=0.60, max_sensitivity=1.5)
if strategies:
    table = v.validate_many(strategies, feats)
    display(table)
    print('Robuste:', int(table['robust'].sum()), '/', len(table))
else:
    print('Nessuna strategia stabile su dati sintetici (comportamento corretto su random walk).')

## ✅ Modulo 5 verificato

Walk-Forward, Monte Carlo, Robustness e Sensitivity con verdetto automatico. Test: `pytest tests/test_validation.py`.

> Su random walk è normale che **nessuna** strategia risulti robusta: è esattamente lo scopo del modulo — non lasciar passare illusioni.

**Prossimo:** Modulo 6 — EA Generator (export MQL4/MQL5 compilabile delle strategie robuste).